In [1]:
! pip install  pdfplumber numpy

In [2]:


import os
import asyncio
from concurrent.futures import ThreadPoolExecutor
import pdfplumber
import logging
import traceback
import nest_asyncio

# Enable nested event loop for Jupyter notebooks
nest_asyncio.apply()

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("pdfplumber_extraction")

# --------- CONFIGURATION (edit these constants) ----------
PDF_SOURCE_DIR = "pdfs_20_2"        # relative to notebook cwd
OUTPUT_DIR = "pdf_plumber_output"      # relative to notebook cwd
MAX_WORKERS = 1                    # reduce if you run out of memory
OFFSET = 0                          # skip first N files
LIMIT = 273                           # 0 = all
EXTRACT_TABLES = True               # extract tables as rows
SEQUENTIAL = False                  # True => process one-by-one (lower mem)
# --------------------------------------------------------

def parse_and_save_pdf(pdf_path: str, out_path: str, extract_tables: bool = True) -> None:
    """
    Blocking worker: open pdf with pdfplumber, collect metadata/pages/tables,
    and write a plain .txt file preserving basic structure.
    """
    try:
        parts = []
        with pdfplumber.open(pdf_path) as pdf:
            metadata = pdf.metadata or {}
            if metadata:
                parts.append("=== DOCUMENT METADATA ===")
                for k, v in metadata.items():
                    if v is not None:
                        parts.append(f"{k}: {v}")
                parts.append("")

            for i, page in enumerate(pdf.pages, start=1):
                try:
                    page_text = page.extract_text() or ""
                except Exception:
                    page_text = ""
                if page_text.strip():
                    parts.append(f"=== PAGE {i} ===")
                    parts.append(page_text.strip())
                    parts.append("")
                if extract_tables:
                    try:
                        tables = page.extract_tables() or []
                    except Exception:
                        tables = []
                    for tnum, table in enumerate(tables, start=1):
                        parts.append(f"--- Table {tnum} on Page {i} ---")
                        for row in table:
                            if row:
                                parts.append(" | ".join(str(c) if c is not None else "" for c in row))
                        parts.append("")

        os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n".join(parts))

        logger.info("Saved: %s", out_path)

    except Exception as e:
        logger.exception("Failed to parse %s: %s\n%s", pdf_path, e, traceback.format_exc())


async def process_folder_async(
    src_folder: str,
    out_folder: str,
    max_workers: int = 2,
    offset: int = 0,
    limit: int = 0,
    extract_tables: bool = True,
    sequential: bool = False,
):
    # Validate folders
    if not os.path.isdir(src_folder):
        raise FileNotFoundError(f"Source folder does not exist: {src_folder}")
    os.makedirs(out_folder, exist_ok=True)

    files = [f for f in os.listdir(src_folder) if f.lower().endswith(".pdf")]
    files.sort()
    if limit > 0:
        files = files[offset: offset + limit]
    else:
        files = files[offset:]

    if not files:
        logger.info("No PDFs to process in %s", src_folder)
        return

    logger.info("Starting: %d files (offset=%d limit=%d) -> %s  sequential=%s workers=%d",
                len(files), offset, limit, out_folder, sequential, max_workers)

    loop = asyncio.get_event_loop()
    processed = 0
    failed = []

    if sequential:
        # Lower memory path: process one-by-one in same process/thread
        for fn in files:
            src = os.path.join(src_folder, fn)
            base = os.path.splitext(fn)[0]
            out_path = os.path.join(out_folder, f"{base}.txt")
            try:
                parse_and_save_pdf(src, out_path, extract_tables)
                processed += 1
                logger.info("✅ [%d/%d] %s", processed, len(files), fn)
            except Exception as e:
                failed.append((fn, str(e)))
                logger.error("❌ [%d/%d] %s -> %s", processed+1, len(files), fn, e)
    else:
        executor = ThreadPoolExecutor(max_workers=max_workers)
        semaphore = asyncio.Semaphore(max_workers)

        async def _worker(fn):
            async with semaphore:
                src = os.path.join(src_folder, fn)
                base = os.path.splitext(fn)[0]
                out_path = os.path.join(out_folder, f"{base}.txt")
                await loop.run_in_executor(executor, parse_and_save_pdf, src, out_path, extract_tables)

        tasks = [asyncio.create_task(_worker(fn)) for fn in files]
        # gather with return_exceptions to let others finish if one fails
        results = await asyncio.gather(*tasks, return_exceptions=True)
        executor.shutdown(wait=True)

        for i, r in enumerate(results):
            fn = files[i]
            if isinstance(r, Exception):
                failed.append((fn, str(r)))
                logger.error("❌ [%d/%d] %s -> %s", i+1, len(files), fn, r)
            else:
                processed += 1
                logger.info("✅ [%d/%d] %s", processed, len(files), fn)

    logger.info("Done. Processed: %d  Failed: %d", processed, len(failed))
    if failed:
        logger.info("Failed files (sample 10): %s", failed[:10])


# ----------------- RUN (edit constants above instead of args) -----------------
# Build absolute paths relative to current working directory
cwd = os.getcwd()
src = os.path.join(cwd, PDF_SOURCE_DIR)
out = os.path.join(cwd, OUTPUT_DIR)

# Run in notebook with await
await process_folder_async(
    src_folder=src,
    out_folder=out,
    max_workers=MAX_WORKERS,
    offset=OFFSET,
    limit=LIMIT,
    extract_tables=EXTRACT_TABLES,
    sequential=SEQUENTIAL,
)


INFO:pdfplumber_extraction:Starting: 273 files (offset=0 limit=273) -> /home/huncho/Workspace/final_project_qna/chat_interface/pdf_plumber_output  sequential=False workers=1
INFO:pdfplumber_extraction:Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pdf_plumber_output/01_gcf-b42-02-add17-funding-proposal-package-fp275.txt
INFO:pdfplumber_extraction:Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pdf_plumber_output/02_gcf-b42-02-add16-funding-proposal-package-fp274.txt
INFO:pdfplumber_extraction:Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pdf_plumber_output/03_gcf-b42-02-add15-funding-proposal-package-fp273.txt
INFO:pdfplumber_extraction:Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pdf_plumber_output/04_gcf-b42-02-add14-funding-proposal-package-fp272_0.txt
INFO:pdfplumber_extraction:Saved: /home/huncho/Workspace/final_project_qna/chat_interface/pdf_plumber_output/05_gcf-b42-02-add13-funding-proposal-package-fp271.txt
INFO